# How CIViC represents Oncogenicity Assertions using GKM

Use this notebook to trace a CIViC oncogenicity assertion from the CIViC user interface into the GA4GH Genomic Knowledge Model (GKM). It accompanies the [CIViC oncogenicity vignette](vignette/).

Start with the CIViC page, then use the GKM Toolkit to inspect its connected GKM records: a VA-Spec assertion and proposition, a Cat-VRS categorical variant with VRS members, evidence lines, source links, and provenance.

!!! tip

    To learn how to explore a published bundle with the GKM Toolkit, see the [Explore mini bundles notebook](../../bundles/explore-civic-bundles.ipynb)

## Starting in CIViC

A CIViC Oncogenic Assertion (Oncogenic AID) summarizes a collection of Evidence Items (EIDs) to provide a classification of somatic variant oncogenicity under the ClinGen/CGC/VICC guidelines [(Horak et al. 2022)](https://pubmed.ncbi.nlm.nih.gov/35101336/).

For this walkthrough, keep two related CIViC concepts separate: **Evidence Items** are the curated records attached to the assertion, while **oncogenicity codes** are the criteria CIViC applies to reach the classification. The CIViC UI displays both, but the current CIViC data model does not link an individual code to an individual Evidence Item.

### CIViC Assertion 202

We will use [CIViC Assertion 202](https://civicdb.org/assertions/202/summary) in this notebook.

The CIViC UI shows that this assertion:

* Classifies **RET M918T as likely oncogenic in medullary thyroid carcinoma** using the ClinGen/CGC/VICC Codes OM1, OS2, OP4, OP1, and OP3
* Classification has been approved by CIViC
* Has 7 attached Evidence Items

![CIViC Assertion 202](civic-assertion-202-ui.png)

The next sections map each part of the UI to GKM.

## From the CIViC assertion page to GKM

### Load the published GKM bundle

CIViCpy creates GKM-compatible records and bundles from CIViC data. This notebook uses the GKM Toolkit to load the published CIViC bundle from the public bundle repository.

!!! note

    The examples read the published CIViC bundle. On first use, the Toolkit may download and cache it locally.

In [1]:
import json

from ga4gh.gkm.bundles import BundleRepository, load_repository_bundle

repository = BundleRepository(refresh=False)
civic_bundle = load_repository_bundle(repository, "civic", refresh=False)
civic_bundle

Bundle(name='civic', collections=17)

### Retrieve the oncogenic assertion from the bundle

We can use the GKM Toolkit to retrieve CIViC AID 202 from the bundle. Its GKM representation is a VA-Spec `VariantOncogenicityStatement`: a knowledge record that evaluates a proposition and stores CIViC's classification.

In [2]:
assertion_id = "civic.aid:202"
assertion = civic_bundle.assertion[assertion_id]
type(assertion)

ga4gh.va_spec.ccv_2022.models.VariantOncogenicityStatement

In [3]:
print(f"CIViC record: {assertion.id}")
print(f"Declared VA-Spec type: {assertion.type}")
print(f"GKM record type: {type(assertion).__name__}")
print(f"CIViC assessment: {assertion.direction} the proposition")
classification = assertion.classification.primaryCoding.code.root
print(f"Classification: {classification}")
print(f"Linked proposition: {assertion.proposition.id}")

CIViC record: civic.aid:202
Declared VA-Spec type: Statement
GKM record type: VariantOncogenicityStatement
CIViC assessment: supports the proposition
Classification: likely oncogenic
Linked proposition: civic.proposition:lGNyTBSVq9ncomifdlwtOERYq7ZM37FX


This record is GKM's counterpart to the CIViC assertion page. Its declared VA-Spec `type` is the general **`Statement`** type. The bundle schema identifies it as the more specific **`VariantOncogenicityStatement`** profile for the ClinGen/CGC/VICC oncogenicity guidelines. The Toolkit uses that profile to create a Python object with oncogenicity-specific fields, including `classification` and `hasEvidenceLines`.

The assertion **supports** the proposition and classifies it as **likely oncogenic**. The Toolkit also follows the bundle reference, so `assertion.proposition` is a typed `VariantOncogenicityProposition`, not a pointer string.

### What claim was classified?

The assertion links to a proposition that holds the variant, disease, gene context, and allele origin. The Toolkit follows that link for you.

In [4]:
proposition = assertion.proposition
variant = proposition.subjectVariant
disease = proposition.objectTumorType
gene = proposition.geneContextQualifier
origin = proposition.alleleOriginQualifier
evidence_lines = assertion.hasEvidenceLines
framework = evidence_lines[0].specifiedBy.name
print(f"Proposition type: {proposition.type}")
print(f"Subject variant: {variant.name}")
print(f"Gene context: {gene.name}")
print(f"Allele origin: {origin.name}")
print(f"Claim: {proposition.predicate} {disease.root.name}")

Proposition type: VariantOncogenicityProposition
Subject variant: RET M918T
Gene context: RET
Allele origin: somatic
Claim: isOncogenicFor Medullary Thyroid Carcinoma


Read the proposition as: **somatic RET M918T is oncogenic for medullary thyroid carcinoma**. The proposition defines the claim. The Statement records CIViC's assessment: *likely oncogenic*. Keeping them separate lets GKM evaluate, cite, and reuse the same claim with its context intact.

## The variation and disease behind the claim

### From a CIViC molecular profile to Cat-VRS and VRS

Precise variation identification requires more than a display name such as `RET M918T`. CIViC groups all sequence contexts under one CIViC Variant. Cat-VRS represents that interpreted category, and its members point to precise VRS Alleles at coding, genomic, and protein levels.

In the assertion UI, we can see the following molecular profile:

![CIViC Assertion Molecular Profile](civic-assertion-202-molecular-profile-ui.png)

Selecting the molecular profile opens:

![CIViC Molecular Profile RET M918T](./civic-molecular-profile-ret-m918t.png)

In [5]:
members = [member.root for member in variant.members]
civic_variant = next(
    mapping.coding
    for mapping in variant.mappings or []
    if mapping.coding.system == "https://civicdb.org/links/variant/"
)

print(f"Categorical variant: {variant.name} ({variant.id})")
print(f"CIViC Variant ID: {civic_variant.id}")
print("VRS Allele IDs by sequence context:")
context_labels = {
    "hgvs.c": "Coding",
    "hgvs.g": "Genomic",
    "hgvs.p": "Protein",
}
protein_allele = next(
    member
    for member in members
    if any(expression.syntax == "hgvs.p" for expression in member.expressions)
)
for member in members:
    context = context_labels[member.expressions[0].syntax]
    print(f"  - {context}: {member.id} ({member.name})")

Categorical variant: RET M918T (civic.mpid:113)
CIViC Variant ID: civic.vid:113
VRS Allele IDs by sequence context:
  - Coding: ga4gh:VA.TZBjEPHhLRYxssQopcOQLWEBQrwzhH3T (NM_020975.4:c.2753T>C)
  - Genomic: ga4gh:VA.ON-Q17mJBYx3unmQ8GiqllzEphxR-Fie (NC_000010.11:g.43121968T>C)
  - Protein: ga4gh:VA.hEybNB_CeKflfFhT5AKOU5i1lgZPP-aS (RET M918T)


#### Full VRS Allele representation

Focus on the protein-level member as one VRS representation.

In [6]:
print(f"\nExample VRS Protein Allele: {protein_allele.id}")
print(f"Name: {protein_allele.name}")
for expression in protein_allele.expressions:
    print(f"  {expression.syntax}: {expression.value}")


Example VRS Protein Allele: ga4gh:VA.hEybNB_CeKflfFhT5AKOU5i1lgZPP-aS
Name: RET M918T
  hgvs.p: NP_065681.1:p.Met918Thr
  hgvs.p: ENSP00000347942.3:p.Met918Thr


This output shows the complete VRS representation of the protein-level member.

In [7]:
print(json.dumps(protein_allele.model_dump(exclude_none=True), indent=2))

{
  "id": "ga4gh:VA.hEybNB_CeKflfFhT5AKOU5i1lgZPP-aS",
  "type": "Allele",
  "name": "RET M918T",
  "digest": "hEybNB_CeKflfFhT5AKOU5i1lgZPP-aS",
  "expressions": [
    {
      "syntax": "hgvs.p",
      "value": "NP_065681.1:p.Met918Thr"
    },
    {
      "syntax": "hgvs.p",
      "value": "ENSP00000347942.3:p.Met918Thr"
    }
  ],
  "location": {
    "id": "ga4gh:SL.oIeqSfOEuqO7KNOPt8YUIa9vo1f6yMao",
    "type": "SequenceLocation",
    "digest": "oIeqSfOEuqO7KNOPt8YUIa9vo1f6yMao",
    "sequenceReference": {
      "type": "SequenceReference",
      "refgetAccession": "SQ.jMu9-ItXSycQsm4hyABeW_UfSNRXRVnl"
    },
    "start": 917,
    "end": 918,
    "sequence": "M"
  },
  "state": {
    "type": "LiteralSequenceExpression",
    "sequence": "T"
  }
}


The CIViC Variant ID connects the CIViC record to its categorical interpretation. Cat-VRS groups its equivalent molecular descriptions into that one interpreted subject, while the VRS Allele IDs identify the precise coding, genomic, and protein-level forms.

The last object is the complete VRS representation for the protein-level context of that CIViC Variant:

- `id` is the computable VRS identifier for this Allele.
- `location` identifies where the variation occurs on the relevant reference sequence.
- `state` identifies what sequence state is observed at that location.
- `expressions` provide familiar human-readable forms, such as protein HGVS.

The coding and genomic member Alleles use the same VRS structure. Together, these records preserve the distinct sequence-level representations grouped under one CIViC Variant.

### Representing the CIViC disease as a mapped concept

In the assertion UI, we can see the following disease:

![CIViC disease as shown in assertion](./civic-assertion-202-diseaase-ui.png)

Selecting the disease opens:

![CIViC Disease Medullary Thyroid Carcinoma](./civic-disease-medullary-thyroid-carcinoma.png)

GKM represents this as a mapped disease concept. It keeps the display name and adds a computable disease identifier:

In [8]:
print("Disease:", disease.root.name)
for mapping in disease.root.mappings or []:
    coding = mapping.coding
    print(f"  {mapping.relation}: {coding.system}{coding.code}")

Disease: Medullary Thyroid Carcinoma
  exactMatch: https://disease-ontology.org/?id=root='DOID:3973'


#### Disease concept JSON

This output shows the full mapped disease concept:

In [9]:
print(json.dumps(disease.model_dump(exclude_none=True), indent=2))

{
  "id": "civic.did:15",
  "conceptType": "Disease",
  "name": "Medullary Thyroid Carcinoma",
  "mappings": [
    {
      "coding": {
        "system": "https://disease-ontology.org/?id=",
        "code": "DOID:3973"
      },
      "relation": "exactMatch"
    }
  ]
}


## The evidence behind the classification

### Representing oncogenicity criteria as Evidence Lines

The CIViC UI shows OM1, OS2, OP4, OP1, and OP3. The published GKM representation contains five VA-Spec Evidence Lines for this criterion-level rationale. The Evidence Items shown in the UI are represented separately, as source URLs later in this notebook.

![CIViC Assertion 202 Summary](civic-assertion-202-summary-ui.png)

In [10]:
print(f"{'Code':<6}{'Method':<32}{'Strength':<14}{'Score'}")
print("-" * 62)
evidence_codes = [
    line.evidenceOutcome.primaryCoding.code.root for line in evidence_lines
]
for line in evidence_lines:
    code = line.evidenceOutcome.primaryCoding.code.root
    method = line.specifiedBy.methodType
    strength = line.strengthOfEvidenceProvided.primaryCoding.code.root
    score = line.scoreOfEvidenceProvided
    print(f"{code:<6}{method:<32}{strength:<14}{score}")

total_score = sum(line.scoreOfEvidenceProvided for line in evidence_lines)
print(f"Total classification score: {total_score}")
evidence_summary = ", ".join(
    f"{line.specifiedBy.methodType.replace('_', '-')} "
    f"({line.evidenceOutcome.primaryCoding.code.root})"
    for line in evidence_lines
)

Code  Method                          Strength      Score
--------------------------------------------------------------
OM1   functional_domain_location      moderate      2
OS2   functional_assay                strong        4
OP4   population_frequency            supporting    1
OP1   computational_prediction        supporting    1
OP3   somatic_hotspot_recurrence      supporting    1
Total classification score: 9


### Why oncogenicity codes and Evidence Items remain separate

The scores are 2 (OM1), 4 (OS2), and 1 each (OP4, OP1, and OP3), totaling 9. The GKM representation uses an Evidence Line for each criterion contributing to the classification. Each line records the criterion, its method, strength, direction, and contribution under the ClinGen/CGC/VICC classification method.

CIViC stores oncogenicity codes on the assertion, but does not link each code to an Evidence Item. Curators may mention an Evidence Item and code in free text, for example `(civic.eid:12709, OS2)`. Descriptions can also include other parenthetical references, such as database versions `(v4.1.0, OP4)`, and curators use different conventions. CIViCpy therefore cannot reliably link Evidence Lines to Evidence Items from these descriptions. Its CIViC-to-GKM representation leaves `EvidenceLine.hasEvidenceItems` empty. A future CIViC model could link each Evidence Line directly to its Evidence Items.

A **CIViC Evidence Item** is a curated record associated with the assertion. CIViCpy retains each Evidence Item URL in a source document referenced by `Statement.reportedIn`. This reflects the current CIViC model: it does not identify which Evidence Item supports each oncogenicity code.

In [11]:
for line in evidence_lines:
    criterion = line.evidenceOutcome.primaryCoding.code.root
    print(criterion, "hasEvidenceItems:", line.hasEvidenceItems)

OM1 hasEvidenceItems: None
OS2 hasEvidenceItems: None
OP4 hasEvidenceItems: None
OP1 hasEvidenceItems: None
OP3 hasEvidenceItems: None


### Distinguishing evidence strength, level, and curator quality

**Evidence strength** is part of an oncogenicity Evidence Line: for example, OM1 is moderate and OS2 is strong. It describes the weight the ClinGen/CGC/VICC method assigns to that criterion.

**Evidence level** and **trust rating / quality** belong to CIViC Evidence Items. Evidence level describes the kind of underlying evidence, while the star rating in the CIViC UI reflects curator confidence in that item. A carefully curated record can still describe preclinical evidence. In the current GKM bundle for this assertion, Evidence Items are retained as URLs, so their UI star ratings are not available as structured GKM quality assessments.

### Linking CIViC Evidence Items to source documents

The CIViC UI lists Evidence Items separately from the oncogenicity codes:

![CIViC Assertion 202 EIDs](./civic-assertion-202-eids-ui.png)

CIViCpy keeps these links in `assertion.reportedIn`. They lead back to the original CIViC Evidence Item and its source context.

In [12]:
evidence_item_documents = [
    document for document in assertion.reportedIn if hasattr(document, "urls")
]
evidence_item_links = []
print("Evidence Item URL -> source document")
for document in evidence_item_documents:
    civic_eid_url = next(url for url in document.urls if "/links/evidence/" in url)
    pmid_url = f"https://pubmed.ncbi.nlm.nih.gov/{document.pmid}/"
    evidence_item_links.append(
        f"[{civic_eid_url}]({civic_eid_url}) ([PMID {document.pmid}]({pmid_url}))"
    )
    print(f"- {civic_eid_url}: PMID:{document.pmid} ({document.id})")
evidence_item_bullets = "\n".join(f"- {item}" for item in evidence_item_links)

Evidence Item URL -> source document
- https://civicdb.org/links/evidence/74: PMID:18073307 (civic.sid:44)
- https://civicdb.org/links/evidence/78: PMID:9839497 (civic.sid:92)
- https://civicdb.org/links/evidence/12711: PMID:29515777 (civic.sid:5458)
- https://civicdb.org/links/evidence/12805: PMID:17108110 (civic.sid:5519)
- https://civicdb.org/links/evidence/11867: PMID:32284345 (civic.sid:4870)
- https://civicdb.org/links/evidence/12709: PMID:9191060 (civic.sid:4953)


Evidence Item URLs lead to CIViC's curated observation records and their source documents. Evidence Lines cite the ClinGen/CGC/VICC guideline instead. This separates documents that report observations from the method that weighs the oncogenicity criteria.

In [13]:
guideline = evidence_lines[0].specifiedBy.reportedIn
print(guideline.name)
print(guideline.title)
print("PMID:", guideline.pmid)

Horak et al., 2022, Genet Med.
Standards for the classification of pathogenicity of somatic variants in cancer (oncogenicity): Joint recommendations of Clinical Genome Resource (ClinGen), Cancer Genomics Consortium (CGC), and Variant Interpretation for Cancer Consortium (VICC)
PMID: 35101336


## Recording CIViC's approval

VA-Spec contributions connect a record to an agent, activity, and date. Assertion 202 records the CIViC organization as the agent for its latest approval review. The current bundle represents this as a `Contribution`; it does not provide a separate provenance-action object.

![CIViC Assertion 202 Approvals](./civic-assertion-202-approvals-ui.png)

In [14]:
approval = next(
    contribution
    for contribution in assertion.contributions or []
    if contribution.activityType.startswith("approval")
)
approval_date = str(approval.date).split(" ")[0]

for contribution in assertion.contributions or []:
    contributor = contribution.contributor
    print(f"{contribution.activityType} by {contributor.name} on {contribution.date}")

approval.last_reviewed by CIViC on 2026-04-16 00:00:00


## Reusing the connected interpretation

The published bundle uses pointers so it does not duplicate records. The Toolkit can expand them into one connected record that another application can use without implementing CIViC's internal data model.

In [15]:
connected = civic_bundle.normalize(assertion)
print(f"Connected record: {connected['id']}")
print(f"Classification: {connected['classification']['primaryCoding']['code']}")
print(f"Variant: {connected['proposition']['subjectVariant']['name']}")
print(f"Disease: {connected['proposition']['objectTumorType']['name']}")
print("Evidence lines:", ", ".join(evidence_codes))
print(f"Approving agent: {connected['contributions'][0]['contributor']['name']}")

Connected record: civic.aid:202
Classification: likely oncogenic
Variant: RET M918T
Disease: Medullary Thyroid Carcinoma
Evidence lines: OM1, OS2, OP4, OP1, OP3
Approving agent: CIViC


### Construct the complete interpretation

The final cell combines the connected records. It lists oncogenicity codes and CIViC Evidence Items separately because the current CIViC model does not identify which item supports which code. Each Evidence Item links to its source PMID.

In [16]:
from IPython.display import Markdown
from IPython.display import display as ipython_display

interpretation = (
    f"**{assertion.id.upper()}:**\n\n"
    f"**{origin.name.capitalize()}** **{variant.name}** is "
    f"**{classification}** for **{disease.root.name}**, evaluated under the "
    f"**{framework}** framework.\n\n"
    "The classification has a score of "
    f"**{total_score}** and is supported by {evidence_summary} evidence. CIViC associates "
    f"the following Evidence Items and source documents with the assertion:\n\n"
    f"{evidence_item_bullets}\n\n"
    "The classification was "
    f"{approval.activityType} by **{approval.contributor.name}** on "
    f"**{approval_date}**."
)
ipython_display(Markdown(interpretation))

**CIVIC.AID:202:**

**Somatic** **RET M918T** is **likely oncogenic** for **Medullary Thyroid Carcinoma**, evaluated under the **ClinGen/CGC/VICC Guidelines for Oncogenicity, 2022** framework.

The classification has a score of **9** and is supported by functional-domain-location (OM1), functional-assay (OS2), population-frequency (OP4), computational-prediction (OP1), somatic-hotspot-recurrence (OP3) evidence. CIViC associates the following Evidence Items and source documents with the assertion:

- [https://civicdb.org/links/evidence/74](https://civicdb.org/links/evidence/74) ([PMID 18073307](https://pubmed.ncbi.nlm.nih.gov/18073307/))
- [https://civicdb.org/links/evidence/78](https://civicdb.org/links/evidence/78) ([PMID 9839497](https://pubmed.ncbi.nlm.nih.gov/9839497/))
- [https://civicdb.org/links/evidence/12711](https://civicdb.org/links/evidence/12711) ([PMID 29515777](https://pubmed.ncbi.nlm.nih.gov/29515777/))
- [https://civicdb.org/links/evidence/12805](https://civicdb.org/links/evidence/12805) ([PMID 17108110](https://pubmed.ncbi.nlm.nih.gov/17108110/))
- [https://civicdb.org/links/evidence/11867](https://civicdb.org/links/evidence/11867) ([PMID 32284345](https://pubmed.ncbi.nlm.nih.gov/32284345/))
- [https://civicdb.org/links/evidence/12709](https://civicdb.org/links/evidence/12709) ([PMID 9191060](https://pubmed.ncbi.nlm.nih.gov/9191060/))

The classification was approval.last_reviewed by **CIViC** on **2026-04-16**.